# `brain_forecast` example usage

End-to-end walkthrough of the library on synthetic data. Mirrors what the CLI does, but exposes every step so you can plug in custom predictors, custom adapters, or custom CV schemes.

## What this notebook covers
1. Build / load a tabular dataset with the required columns (`sub`, `start`, `cohort` + features + targets)
2. Configure the FeatureBundle and CV
3. Run the experiment across multiple predictors and horizons
4. Aggregate scores and plot the horizon-curve benchmark figure

In [ ]:
import logging
import warnings
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
warnings.filterwarnings('ignore')

## 1. Build a synthetic dataset

In real usage you would `pd.read_parquet(...)` your own file. The library only requires the three metadata columns `sub`, `start` (seconds), `cohort` (movie identifier) plus your feature and target columns.

The synthetic dataset here has the same shape as the user's real data:
- 3 movies (cohorts) with different subject counts → tests cohort-stratified CV with LOSO fallback
- A few movie features (`mov_*`) and brain features (`umap0/5`)
- DFC connection targets (`UDFC_*`)

In [ ]:
np.random.seed(42)
TR = 1.5  # seconds
movie_specs = {'500daysofsummer': 12, 'pulpfiction': 4, 'citizenfour': 12}
rows = []
for cohort, n_subs in movie_specs.items():
    for sub_idx in range(n_subs):
        sub_id = f'{cohort}_sub{sub_idx:03d}'
        T = 200
        mov1 = np.sin(np.arange(T) * 0.1 + sub_idx) + np.random.randn(T) * 0.1
        mov2 = np.cos(np.arange(T) * 0.07) + np.random.randn(T) * 0.1
        ubrain = 0.5 * mov1 + np.random.randn(T) * 0.3
        udfc1 = 0.6 * np.roll(mov1, 3) + 0.3 * ubrain + np.random.randn(T) * 0.2
        udfc2 = 0.5 * np.roll(mov2, 3) + np.random.randn(T) * 0.3
        for t in range(T):
            rows.append({
                'sub': sub_id, 'start': t * TR, 'cohort': cohort,
                'mov_v1': mov1[t], 'mov_audio': mov2[t],
                'umap0/5': ubrain[t],
                'UDFC_1': udfc1[t], 'UDFC_2': udfc2[t],
            })
df = pd.DataFrame(rows)
df.to_parquet('/tmp/example_data.parquet')
print(f'{len(df):,} rows, {df.sub.nunique()} subjects, {df.cohort.nunique()} cohorts')
df.head()

## 2. Load into a FeatureBundle

In [ ]:
from brain_forecast.data import load_bundle

bundle = load_bundle(
    path='/tmp/example_data.parquet',
    feature_cols=['mov_v1', 'mov_audio', 'umap0/5'],
    target_cols=['UDFC_1', 'UDFC_2'],
    task_type='regression',
)
print(f'Subjects: {bundle.n_subjects()}')
print(f'Cohorts:  {list(bundle.cohorts())}')

## 3. Configure CV — stratified subject-out

Each fold's test set draws subjects from every movie proportionally. Cohorts smaller than `loso_threshold` switch to leave-one-subject-out automatically.

In [ ]:
from brain_forecast.cv import StratifiedSubjectOutCV

cv = StratifiedSubjectOutCV(k_default=4, loso_threshold=6, random_state=0)
for fold_idx, (train_subs, test_subs) in enumerate(cv.split(bundle)):
    composition = bundle.df[bundle.df['sub'].isin(test_subs)].groupby('cohort')['sub'].nunique().to_dict()
    print(f'fold {fold_idx}: {len(train_subs)} train / {len(test_subs)} test (test composition: {composition})')

## 4. Run experiment — three benchmarks across horizons

For the v0 smoke test, skip TFT (needs GPU and pytorch-forecasting installed) and just run Persistence, Moving Average, AR. Add `{'name': 'tft', 'kwargs': {...}}` when ready.

In [ ]:
from brain_forecast.features import TabularAdapter
from brain_forecast.evaluation import run_experiment

results = run_experiment(
    bundle=bundle,
    predictor_specs=[
        {'name': 'persistence'},
        {'name': 'moving_average', 'kwargs': {'k': 5}},
        {'name': 'ar', 'kwargs': {'p': 5}},
    ],
    horizons_min=[0, 0.25, 0.5, 1.0],
    cv=cv,
    tabular_adapter=TabularAdapter(
        ops=['lag', 'rolling', 'target_history'],
        k_lag=3, rolling_window=10, target_history_lags=5,
    ),
    output_dir='/tmp/example_run',
)
results.head()

## 5. Aggregate and plot

In [ ]:
from brain_forecast.reporting import aggregate_scores, plot_horizon_curves, plot_per_movie_heatmap

agg = aggregate_scores(results, metric='r2_mean', by=['predictor', 'horizon_min'])
agg

In [ ]:
plot_horizon_curves(results, metric='r2_mean', title='R² vs forecast horizon');

In [ ]:
plot_horizon_curves(results, metric='r2_mean', per_cohort=True, title='Per-cohort breakdown');

In [ ]:
plot_per_movie_heatmap(results, metric='r2_mean');

## Next steps

- Replace synthetic data with real parquet on disk.
- Switch `target_cols` to your actual ROI / DFC connection / brain-state column names; set `task_type` accordingly.
- Add the TFT predictor to `predictor_specs` once you have a GPU and `pytorch-forecasting` installed.
- Run the same harness from the CLI on a YAML config for reproducibility:
  ```bash
  python -m brain_forecast run --config configs/example.yaml
  ```